# Interval conformal (split)
**tabular › regression_advanced** · notebook `conformal_interval`

**Kapan dipakai:** Interval dengan jaminan coverage pada sampel hingga untuk model apa pun, hanya dari residual data kalibrasi.

**Jebakan umum:** Menganggap data kalibrasi dan test exchangeable (tidak ada pergeseran). Lebar interval konstan; tidak adaptif terhadap heteroskedastisitas.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular/regression/diamonds_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular/regression/diamonds_test.csv"      # None kalau tidak ada file test
TARGET     = "price"
ID_COL     = 'id'
DROP_COLS  = []
GROUP_COL  = None                      # isi bila ada grup agar tidak bocor
SCORING    = "neg_root_mean_squared_error"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "conformal_interval"
TASK       = "regression"
KIND       = "tree"
OUT = out_dir("tabular", "regression_advanced", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
y = df[TARGET].to_numpy(dtype=float)
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Model dasar

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.base import clone
lgbm = LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbose=-1)
def fit_eval(name, model_, kind="tree", ytr_=None, store=None):
    pp = build_pipeline(model_, NUM, CAT, kind=kind).fit(Xtr, ytr if ytr_ is None else ytr_)
    pr = pp.predict(Xva)
    if store is not None: store[name] = pr
    return {"model": name, **regression_metrics(yva, pr)}

## 5. Split conformal
Bagi training jadi *fit* dan *kalibrasi*. Interval = prediksi ± kuantil residual mutlak pada data kalibrasi. Tanpa asumsi distribusi.

In [ ]:
from sklearn.model_selection import train_test_split
from superdb.tabular_extra_b import split_conformal_halfwidth, interval_coverage
Xf, Xc_, yf, yc = train_test_split(Xtr, ytr, test_size=0.25, random_state=SEED)
pp = build_pipeline(clone(lgbm), NUM, CAT, kind="tree").fit(Xf, yf)
res_cal = yc - pp.predict(Xc_); pv = pp.predict(Xva)
rows = []
for alpha in (0.05, 0.1, 0.2, 0.3):
    h = split_conformal_halfwidth(res_cal, alpha)
    cov, wd = interval_coverage(yva, pv - h, pv + h)
    # varian relatif: lebar ∝ level prediksi (cocok untuk target positif dengan galat proporsional)
    rel = split_conformal_halfwidth(np.abs(res_cal) / np.maximum(pp.predict(Xc_), 1e-9), alpha)
    cov_r, wd_r = interval_coverage(yva, pv * (1 - rel), pv * (1 + rel))
    rows.append({"nominal": 1 - alpha, "coverage_konstan": cov, "lebar_konstan": wd, "coverage_relatif": cov_r, "lebar_relatif": wd_r})
display(pd.DataFrame(rows).round(3))

Coverage nyata harus ≈ nominal pada data yang exchangeable. Interval **relatif** lebih sempit untuk nilai kecil dan lebih lebar untuk nilai besar, biasanya lebih masuk akal untuk harga/jumlah.

In [ ]:
h = split_conformal_halfwidth(res_cal, 0.1)
metrics = {**regression_metrics(yva, pv), "halfwidth_90": h, "coverage_90": interval_coverage(yva, pv - h, pv + h)[0]}
display(show_table(metrics))

In [ ]:
try:
    save_json({"method": MODEL_KEY, "valid": res.reset_index().to_dict("records") if "res" in dir() else metrics}, OUT / "metrics.json")
except NameError:
    save_json({"method": MODEL_KEY, "valid": metrics}, OUT / "metrics.json")
print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan metrik pada skala yang dipakai juri lomba, bukan hanya RMSE.
- Hasil terbaik biasanya digabung ke ensemble (lihat `notebooks/tabular/ensembling_regression/`).